In [1]:
import pandas as pd
import numpy as np

!pip install -q lets-plot
from lets_plot import *
LetsPlot.setup_html()

In [2]:
# 1. Parameters and Setups

# simulation settings

np.random.seed(2026)

# economic and utility
wtp = 150000          # Lambda ($/QALY)
qaly_no_event = 0.8
qaly_event = 0.2
cost_sc = 50000
cost_nt = 60000

# true values
p_sc = 0.3

# Variance of theta function
def get_var_theta(theta, n_sc, n_nt, p_sc=p_sc):
    rr = np.exp(theta)
    p_nt = p_sc * rr

    a = n_sc * p_sc
    var_sc = 1/a - 1/n_sc
    c = n_nt * p_nt
    var_nt = 1/c - 1/n_nt

    return var_nt + var_sc

# study designs
## policy 1: trial
trial_cost=5*1e6
n_trial = 400
n_trial_sc = 0.5*n_trial
n_trial_nt = 0.5*n_trial
tau_trial = 4 # years to results

## polity 2: RWE
bias_b = 0.3
n_rwe=4000
n_rwe_sc=0.5*n_rwe
n_rwe_nt=0.5*n_rwe
delta_tau_rwe=2 # 2 years advance in time

# population
I_t = 10000 # annual cohort
total_yrs = 10 #  time horizon
discount_rate = 0.03
update_rate = 0.3

In [3]:
# 2. Helper Functions

# Posterior update (normal-normal)
def get_posterior_mean_variance(prior_mean, prior_var, data_mean, data_var):

    # Normal-Normal Conjugate
    prior_prec = 1.0 / prior_var
    data_prec = 1.0 / data_var
    post_prec = prior_prec + data_prec

    post_var = 1.0 / post_prec
    post_mean = post_var * (prior_mean * prior_prec + data_mean * data_prec)

    return post_mean, post_var

  # Net benefit
def calculate_net_benefit(theta, cost, qaly_no_event=qaly_no_event, qaly_event=qaly_event, p_sc=p_sc):
    """Calculates NHB = E[QALYs] - Cost/WTP"""
    prob_event = np.exp(theta) * p_sc
    expected_qaly = (1 - prob_event)*qaly_no_event + prob_event*qaly_event
    return expected_qaly - cost/wtp

# Figure 1: Population-level expected net benefit under alternative CED policies as a funciton of RWE bias

In [4]:
scaler=1.3

# Custom colors
colors = {
    'Policy 1: RCT-based CED': '#E41A1B',
    'Policy 2: RWE-based CED': '#377EB8'
}

# Add font family configuration
times_new_roman_theme = theme(
    # title=element_text(size=12*scaler, family="Times New Roman"),
    axis_text_x=element_text(size=10*scaler, family="Times New Roman", angle=0),
    axis_text_y=element_text(size=10*scaler, family="Times New Roman"),
    legend_text=element_text(size=11*scaler, family="Times New Roman"),
    legend_position="bottom",  # Move legend to the bottom
    legend_direction="horizontal",
    legend_title=element_text(size=12*scaler, family="Times New Roman"),
    plot_title=element_text(size=16, family="Times New Roman", face = "bold"),
    plot_subtitle=element_text(size=14, family="Times New Roman"),
    axis_title_x=element_text(size=12*scaler, family="Times New Roman"),
    axis_title_y=element_text(size=12*scaler, family="Times New Roman"),
    text=element_text(size=8, family="Times New Roman")
)

## Base True RR = 0.63

In [5]:
# Figure 1: Expected Net Benefits under RWE bias
# Data Preparation
n_psa = 1000
n_inner = 100000

rr_true = 0.63
bias = 1

In [6]:
# ============================================================
# Probabilistic Sensitivity Analysis: Per-Draw Nested Benefit
# ============================================================

def calculate_nested_benefit_per_draw(rr_prior_mean, bias_b=1.0, n_psa=n_psa, n_inner=n_inner):
    """
    Same logic as calculate_nested_benefit, but returns per-draw arrays
    instead of scalar averages.

    Parameters
    ----------
    rr_prior_mean : float
        Prior mean of the risk ratio (e.g. 0.63).
    bias_b : float
        RWE bias factor B (1.0 = no bias).
    n_psa : int
        Number of outer Monte Carlo draws (true θ draws from prior).
    n_inner : int
        Number of inner Monte Carlo draws (posterior samples per outer draw).

    Returns
    -------
    dict with arrays of length n_psa:
        theta_draws        – drawn true θ values
        nb_nt_per_draw     – NB of NT at each true θ (for pre-disclosure periods)
        payoff_trial       – per-draw realized payoff after trial-informed decision
        payoff_rwe         – per-draw realized payoff after RWE-informed decision
        nb_sc              – scalar NB of standard of care
    """
    theta_prior_mean = np.log(rr_prior_mean)
    theta_prior_var = get_var_theta(
        p_sc=p_sc,
        theta=theta_prior_mean,
        n_sc=100,
        n_nt=100)

    # Net Benefit of SC (constant)
    nb_sc = calculate_net_benefit(theta=0, cost=cost_sc)

    # --- Outer MC: draw n_psa true θ values from the prior ---
    theta_draws = np.random.normal(theta_prior_mean, np.sqrt(theta_prior_var), n_psa)

    # Per-draw NB of NT at the true θ (used for pre-disclosure periods)
    nb_nt_per_draw = calculate_net_benefit(theta_draws, cost_nt)

    # ===================== Policy 1: Trial =====================
    # 1. Generate unbiased trial data — Eq [2]
    var_trial = get_var_theta(theta_draws, n_trial_sc, n_trial_nt)
    X_bar = np.random.normal(theta_draws, np.sqrt(var_trial))

    # 2. Posterior belief given X_bar — Eq [3]
    post_mean_trial, post_var_trial = get_posterior_mean_variance(
        prior_mean=theta_prior_mean,
        prior_var=theta_prior_var,
        data_mean=X_bar,
        data_var=var_trial
    )

    # 3. Inner MC: E[NB_NT | X_bar] via posterior samples — Eq [4]
    dec_samples_trial = np.random.normal(
        loc=post_mean_trial[:, np.newaxis],
        scale=np.sqrt(post_var_trial[:, np.newaxis]),
        size=(n_psa, n_inner)
    )
    expected_nb_nt_trial = np.mean(
        calculate_net_benefit(dec_samples_trial, cost_nt), axis=1
    )

    # 4. Decision & payoff
    choose_nt_trial = expected_nb_nt_trial > nb_sc
    payoff_trial = np.where(choose_nt_trial, expected_nb_nt_trial, nb_sc)

    # ===================== Policy 2: RWE =====================
    # 1. Counterfactual unbiased estimate — Eq [5]
    hypo_var_trial = get_var_theta(theta_draws, n_rwe_sc, n_rwe_nt)
    hypo_X_bar = np.random.normal(theta_draws, np.sqrt(hypo_var_trial))

    # 2. Biased RWE data — Eq [8]-[9]: R_bar | θ ~ N(θ + log(B), σ²_R)
    biased_theta = hypo_X_bar + np.log(bias_b)
    var_rwe = get_var_theta(biased_theta, n_rwe_sc, n_rwe_nt)
    R_bar = np.random.normal(biased_theta, np.sqrt(var_rwe))

    # 3a. Biased posterior (drives the decision) — Eq [10]
    post_mean_rwe, post_var_rwe = get_posterior_mean_variance(
        prior_mean=theta_prior_mean,
        prior_var=theta_prior_var,
        data_mean=R_bar,
        data_var=var_rwe
    )

    # 3b. True posterior from counterfactual unbiased data — Eq [6]
    post_mean_true_rwe, post_var_true_rwe = get_posterior_mean_variance(
        prior_mean=theta_prior_mean,
        prior_var=theta_prior_var,
        data_mean=hypo_X_bar,
        data_var=hypo_var_trial
    )

    # 4. Inner MC for decision (biased posterior)
    dec_samples_rwe = np.random.normal(
        loc=post_mean_rwe[:, np.newaxis],
        scale=np.sqrt(post_var_rwe[:, np.newaxis]),
        size=(n_psa, n_inner)
    )
    expected_nb_nt_rwe = np.mean(
        calculate_net_benefit(dec_samples_rwe, cost_nt), axis=1
    )

    # 5. Inner MC for realized payoff (true posterior)
    dec_samples_true_rwe = np.random.normal(
        loc=post_mean_true_rwe[:, np.newaxis],
        scale=np.sqrt(post_var_true_rwe[:, np.newaxis]),
        size=(n_psa, n_inner)
    )
    expected_nb_nt_true_rwe = np.mean(
        calculate_net_benefit(dec_samples_true_rwe, cost_nt), axis=1
    )

    # 6. Decision from biased, payoff from true — Eq [11]-[12]
    choose_nt_rwe = expected_nb_nt_rwe > nb_sc
    payoff_rwe = np.where(choose_nt_rwe, expected_nb_nt_true_rwe, nb_sc)

    return {
        'theta_draws': theta_draws,
        'nb_nt_per_draw': nb_nt_per_draw,
        'payoff_trial': payoff_trial,
        'payoff_rwe': payoff_rwe,
        'nb_sc': nb_sc
    }

In [7]:
# ============================================================
# PSA: Population-Level Policy NB Per Draw & Crossover RR
# ============================================================

def run_psa(rr_prior_mean=0.63, bias_b=1.0, n_psa=n_psa, n_inner=n_inner,
            delta_tau_rwe=delta_tau_rwe, trial_cost=trial_cost):
    """
    Probabilistic sensitivity analysis.
    For each of n_psa draws of the true RR from the prior:
      - compute population-level NB under Policy 1 (RCT-CED) and Policy 2 (RWE-CED)
    Then sort by RR and find the crossover where the two policies are indifferent.

    Returns
    -------
    df : pd.DataFrame
        Columns: rr, policy_1_nb, policy_2_nb, diff  (sorted by rr ascending)
    crossover_rr : float or None
        Interpolated RR where Policy 2 NB = Policy 1 NB (first crossing from above).
    prop_policy1 : float
        Proportion of simulations where Policy 1 NB > Policy 2 NB.
    """
    # --- 1. Per-draw nested benefits ---
    res = calculate_nested_benefit_per_draw(
        rr_prior_mean=rr_prior_mean,
        bias_b=bias_b,
        n_psa=n_psa,
        n_inner=n_inner
    )

    theta_draws    = res['theta_draws']
    nb_nt_per_draw = res['nb_nt_per_draw']   # NB of NT at each true Ͱ
    payoff_trial   = res['payoff_trial']      # per-draw payoff after trial decision
    payoff_rwe     = res['payoff_rwe']        # per-draw payoff after RWE decision
    nb_sc          = res['nb_sc']             # scalar

    # --- 2. Pre-compute discounted populations (deterministic) ---
    # Policy 1
    discounted_pop_waiting = np.sum(
        [I_t / ((1 + discount_rate)**t) for t in range(1, tau_trial + 1)]
    )
    discounted_pop_after_trial = np.sum(
        [I_t / ((1 + discount_rate)**t) for t in range(tau_trial + 1, total_yrs)]
    )

    # Policy 2
    cutoff_rwe = tau_trial - delta_tau_rwe
    discounted_pop_before_rwe = np.sum(
        [I_t / ((1 + discount_rate)**t) for t in range(0, cutoff_rwe + 1)]
    )
    discounted_pop_after_rwe = np.sum(
        [I_t / ((1 + discount_rate)**t) for t in range(cutoff_rwe + 1, total_yrs)]
    )

    # --- 3. Population-level NB per draw (vectorized) ---

    # Policy 1: RCT-based CED  — Eq [13]-[16]
    trial_pop_nb = n_trial_sc * nb_sc + n_trial_nt * nb_nt_per_draw - trial_cost / wtp
    before_trial_pop_nb = (discounted_pop_waiting + I_t - n_trial) * nb_sc
    expected_nb_after_trial = update_rate * payoff_trial + (1 - update_rate) * nb_sc
    after_trial_pop_nb = discounted_pop_after_trial * expected_nb_after_trial

    policy_1 = trial_pop_nb + before_trial_pop_nb + after_trial_pop_nb

    # Policy 2: RWE-based CED  — Eq [17]-[19]
    expected_nb_before_rwe = update_rate * nb_nt_per_draw + (1 - update_rate) * nb_sc
    policy_2_before = discounted_pop_before_rwe * expected_nb_before_rwe

    expected_nb_after_rwe = update_rate * payoff_rwe + (1 - update_rate) * nb_sc
    policy_2_after = discounted_pop_after_rwe * expected_nb_after_rwe

    policy_2 = policy_2_before + policy_2_after

    # --- 4. Sort by RR ascending ---
    rr_draws = np.exp(theta_draws)
    sort_idx = np.argsort(rr_draws)

    rr_sorted = rr_draws[sort_idx]
    p1_sorted = policy_1[sort_idx]
    p2_sorted = policy_2[sort_idx]
    diff_sorted = p2_sorted - p1_sorted   # positive → RWE preferred

    # --- 5. Find crossover(s) via sign change ---
    sign_changes = np.where(np.diff(np.sign(diff_sorted)))[0]

    crossover_rrs = []
    for idx in sign_changes:
        # Linear interpolation between idx and idx+1
        rr_lo, rr_hi = rr_sorted[idx], rr_sorted[idx + 1]
        d_lo, d_hi = diff_sorted[idx], diff_sorted[idx + 1]
        if d_hi != d_lo:
            rr_cross = rr_lo + (rr_hi - rr_lo) * (-d_lo) / (d_hi - d_lo)
            crossover_rrs.append(rr_cross)

    crossover_rr = crossover_rrs[0] if crossover_rrs else None

    # --- 6. Calculate Proportion Policy 1 > Policy 2 ---
    # Policy 1 is preferred when p1 > p2, or diff < 0
    prop_policy1 = np.mean(p1_sorted > p2_sorted)

    # --- 7. Build output DataFrame ---
    df = pd.DataFrame({
        'rr': rr_sorted,
        'policy_1_nb': p1_sorted,
        'policy_2_nb': p2_sorted,
        'diff': diff_sorted
    })

    return df, crossover_rr, crossover_rrs, prop_policy1

In [8]:
# --- Run PSA: B=1, prior RR=0.63, 1000 draws ---
np.random.seed(2026)
df_psa, crossover_rr, crossover_rrs, prop_p1 = run_psa(
    rr_prior_mean=0.63,
    bias_b=1.0,
    n_psa=n_psa,
    n_inner=n_inner
)

# --- Print results ---
print(f"Number of MC draws: {len(df_psa)}")
print(f"RR range:  [{df_psa['rr'].min():.4f},  {df_psa['rr'].max():.4f}]")
print(f"Number of crossover(s) found: {len(crossover_rrs)}")
print(f"Proportion Policy 1 Preferred: {prop_p1:.2%}")

if len(crossover_rrs) > 0:
    rr_min_cross = min(crossover_rrs)
    rr_max_cross = max(crossover_rrs)

    print(f"\n>>> Min Crossover RR: {rr_min_cross:.4f} <<<")
    print(f">>> Max Crossover RR: {rr_max_cross:.4f} <<<")
    print(f"\nInterpretation:")
    print(f"  RR < {rr_min_cross:.4f}  → RWE-CED clearly preferred (earlier access to effective NT)")
    print(f"  RR > {rr_max_cross:.4f}  → RCT-CED clearly preferred (early coverage of ineffective NT hurts)")
    print(f"  {rr_min_cross:.4f} < RR < {rr_max_cross:.4f}  → indifference zone (mixed due to simulation noise)")

Number of MC draws: 1000
RR range:  [0.2714,  1.5954]
Number of crossover(s) found: 91
Proportion Policy 1 Preferred: 53.10%

>>> Min Crossover RR: 0.5166 <<<
>>> Max Crossover RR: 0.6907 <<<

Interpretation:
  RR < 0.5166  → RWE-CED clearly preferred (earlier access to effective NT)
  RR > 0.6907  → RCT-CED clearly preferred (early coverage of ineffective NT hurts)
  0.5166 < RR < 0.6907  → indifference zone (mixed due to simulation noise)


In [9]:
# --- PSA Plot: Policy NB vs Drawn True RR ---

# Reshape for lets-plot
df_plot = pd.DataFrame({
    'RR': np.concatenate([df_psa['rr'].values, df_psa['rr'].values]),
    'NB (kQALYs)': np.concatenate([df_psa['policy_1_nb'].values / 1000,
                                    df_psa['policy_2_nb'].values / 1000]),
    'Policy': (['Policy 1: RCT-based CED'] * len(df_psa) +
               ['Policy 2: RWE-based CED'] * len(df_psa))
})

p = (ggplot(df_plot, aes(x='RR', y='NB (kQALYs)', color='Policy'))
     + geom_point(alpha=0.3, size=1.5)
     + scale_color_manual(values=colors)
     + labs(
         title='PSA: Policy NB vs Drawn True RR (B = 1, No Bias)',
         subtitle=f'Proportion Policy 1 Preferred: {prop_p1:.2%}',
         x='Drawn True Risk Ratio (RR)',
         y='Population-Level Expected NB (kQALYs)',
         color=''
     )
     + times_new_roman_theme
)

# Add min and max crossover lines
if len(crossover_rrs) > 0:
    rr_min_cross = min(crossover_rrs)
    rr_max_cross = max(crossover_rrs)
    y_top = df_plot['NB (kQALYs)'].max()

    # Min crossover (dashed)
    p = p + geom_vline(xintercept=rr_min_cross, linetype='dashed', color='black', size=0.8)
    # True
    p = p + geom_vline(xintercept=rr_true, linetype='solid', color='black', size=0.8)
    # Max crossover (dashed)
    p = p + geom_vline(xintercept=rr_max_cross, linetype='dashed', color='black', size=0.8)

    # Labels
    p = p + geom_label(
        aes(x='x', y='y', label='label'),
        data=pd.DataFrame({
            'x': [rr_min_cross - 0.03, rr_true, rr_max_cross + 0.03],
            'y': [y_top * 0.99, y_top*0.87, y_top * 0.99],
            'label': [f'Min RR = {rr_min_cross:.2f}', f'Point RR = {rr_true:.2f}',
                      f'Max RR = {rr_max_cross:.2f}']
        }),
        color='black', size=5, family='Times New Roman',
        inherit_aes=False
    )

p